In [1]:
import torch
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU'
!nvidia-smi -L
import os, time, random, shutil, sys, importlib
from pathlib import Path
import numpy as np
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')

CLASSES = ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam']
NUM_CLASSES, IMG_SIZE, MODEL_NAME = 5, 224, 'convnext_tiny'
EPOCHS, BATCH, LR, WD, SMOOTH, SEED = 20, 32, 2e-4, 0.05, 0.1, 13
IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
SRC = '/content/drive/MyDrive/train'
OUT_DIR = '/content/drive/MyDrive/studio2_out/final'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs('submission', exist_ok=True)

assert all(os.path.isdir(f'{SRC}/{c}') for c in CLASSES), f'Folder kelas tidak ada di {SRC}: {os.listdir(SRC) if os.path.isdir(SRC) else "folder tidak ada"}'
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device('cuda')

GPU 0: Tesla T4 (UUID: GPU-9318960a-78ea-6abb-debb-f38c1d27c261)
Mounted at /content/drive


In [2]:
TRAIN_DIR = '/content/train_local'
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
for c in CLASSES: shutil.copytree(f'{SRC}/{c}', f'{TRAIN_DIR}/{c}')

def shrink(im, s=256):
    im = im.convert('RGB'); w, h = im.size; r = s / min(w, h)
    return im.resize((max(s, round(w * r)), max(s, round(h * r))), Image.BILINEAR)

images, labels = [], []
for ci, c in enumerate(CLASSES):
    for p in sorted(Path(TRAIN_DIR, c).iterdir()):
        if p.suffix.lower() in IMG_EXT:
            try: images.append(shrink(Image.open(p))); labels.append(ci)
            except Exception as e: print('skip', p.name, e)
labels = np.array(labels)
print(len(images), np.bincount(labels, minlength=NUM_CLASSES))
assert len(images) == 2753, f'Jumlah gambar {len(images)} != 2753 (jumlah saat CV). Cek folder train.'


rng = np.random.default_rng(0); sanity_ids = rng.choice(len(images), 200, replace=False)
shutil.rmtree('my_val', ignore_errors=True)
for i in sanity_ids:
    d = Path('my_val') / CLASSES[labels[i]]; d.mkdir(parents=True, exist_ok=True)
    images[i].save(d / f'{i}.png')

2753 [523 521 365 673 671]


In [3]:
MODEL_PY = '''import torch
import torch.nn as nn
from torchvision import models, transforms

NUM_CLASSES, IMG_SIZE = 5, __IMG_SIZE__
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = models.__MODEL_NAME__(weights=None)
        self.net.classifier[2] = nn.Linear(768, NUM_CLASSES)

    def forward(self, x):
        out = self.net(x)
        if not self.training:
            out = (out + self.net(torch.flip(x, dims=[3]))) / 2
        return out


def get_model():

    return Net()


def get_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
'''.replace('__IMG_SIZE__', str(IMG_SIZE)).replace('__MODEL_NAME__', MODEL_NAME)
open('submission/model.py', 'w').write(MODEL_PY)
open('submission/requirements.txt', 'w').write('')
sys.path.insert(0, os.getcwd())
import submission.model as sm
importlib.reload(sm)
MEAN, STD = sm.MEAN, sm.STD

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

pre = {k: v for k, v in models.ConvNeXt_Tiny_Weights.DEFAULT.get_state_dict(progress=True).items() if not k.startswith('classifier.2')}
torch.manual_seed(1234)
model = sm.get_model(); print(model.net.load_state_dict(pre, strict=False)); model.to(device)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 173MB/s]


_IncompatibleKeys(missing_keys=['classifier.2.weight', 'classifier.2.bias'], unexpected_keys=[])


Net(
  (net): ConvNeXt(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
      )
      (1): Sequential(
        (0): CNBlock(
          (block): Sequential(
            (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
            (1): Permute()
            (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
            (3): Linear(in_features=96, out_features=384, bias=True)
            (4): GELU(approximate='none')
            (5): Linear(in_features=384, out_features=96, bias=True)
            (6): Permute()
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): CNBlock(
          (block): Sequential(
            (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
            (1): Permute()
            (2): LayerNorm((96,), eps=1e-06,

In [4]:
class DS(Dataset):
    def __init__(s, ids, tf): s.ids, s.tf = ids, tf
    def __len__(s): return len(s.ids)
    def __getitem__(s, k):
        i = s.ids[k]; return s.tf(images[i]), int(labels[i])

ids = np.arange(len(images))
cnt = np.bincount(labels, minlength=NUM_CLASSES)
sw = (1.0 / np.sqrt(cnt))[labels]
sampler = torch.utils.data.WeightedRandomSampler(sw, len(ids), replacement=True)
W = min(4, os.cpu_count() or 2)
tl = DataLoader(DS(ids, train_tf), BATCH, sampler=sampler, num_workers=W, drop_last=True, pin_memory=True)

crit = nn.CrossEntropyLoss(label_smoothing=SMOOTH)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
steps, warm = EPOCHS * len(tl), len(tl)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, steps - warm))))
scaler = torch.amp.GradScaler('cuda')
seed_all(SEED)

for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0
    for x, y in tl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda'):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    print(f'ep {ep+1}/{EPOCHS} loss {tot/len(tl):.4f} {time.time()-t0:.0f}s')
    if (ep + 1) % 5 == 0:
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, f'{OUT_DIR}/ckpt_ep{ep+1}.pth')

torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()}, 'submission/model.pth')
print('selesai | model.pth: %.1f MB' % (os.path.getsize('submission/model.pth') / 1e6))

ep 1/20 loss 0.9901 31s
ep 2/20 loss 0.5820 24s
ep 3/20 loss 0.5350 23s
ep 4/20 loss 0.5173 24s
ep 5/20 loss 0.5032 25s
ep 6/20 loss 0.4838 25s
ep 7/20 loss 0.4892 23s
ep 8/20 loss 0.4683 24s
ep 9/20 loss 0.4565 24s
ep 10/20 loss 0.4468 24s
ep 11/20 loss 0.4295 24s
ep 12/20 loss 0.4427 24s
ep 13/20 loss 0.4238 24s
ep 14/20 loss 0.4291 24s
ep 15/20 loss 0.4134 24s
ep 16/20 loss 0.4154 24s
ep 17/20 loss 0.4064 24s
ep 18/20 loss 0.4103 24s
ep 19/20 loss 0.4098 24s
ep 20/20 loss 0.4147 24s
selesai | model.pth: 55.7 MB


In [5]:
shutil.copy('/content/drive/MyDrive/evaluate.py', 'evaluate.py')
!python evaluate.py --submission ./submission --data ./my_val --out final_eval.json

import json
r = json.load(open('final_eval.json'))
print(f"\nSANITY (bocor): Macro-F1 {r['macro_f1']:.4f} | n={r['n_images']} | {r['seconds']}s")
assert r['macro_f1'] >= 0.98, 'Sanity < 0.98 -> model mungkin rusak, JANGAN dipakai. Pakai model lama di Drive (studio2_out/model.pth).'
for f in ('model.pth', 'model.py', 'requirements.txt'):
    shutil.copy(f'submission/{f}', f'{OUT_DIR}/{f}')
print('Salinan final di', OUT_DIR)

device=cuda  gambar=200

Macro-F1 : 1.0000
Akurasi  : 1.0000
  F1 botol_plastik   1.0000
  F1 gelas_plastik   1.0000
  F1 kertas_kardus   1.0000
  F1 saset_kemasan   1.0000
  F1 styrofoam       1.0000
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [35, 0, 0, 0, 0]
  gelas_plastik   [0, 34, 0, 0, 0]
  kertas_kardus   [0, 0, 27, 0, 0]
  saset_kemasan   [0, 0, 0, 49, 0]
  styrofoam       [0, 0, 0, 0, 55]

SANITY (bocor): Macro-F1 1.0000 | n=200 | 3.4s
Salinan final di /content/drive/MyDrive/studio2_out/final


In [6]:
from google.colab import files
for f in ('submission/model.pth', 'submission/model.py', 'submission/requirements.txt'):
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>